# EDEN-00 — Dataset inventory & acquisition
**Project:** Environmental Digital Twin for Ecosystem Intelligence and Navigation (EDEN)  
**Phase:** M1 / Data engineering / Notebook 00 of 15 (16 notebooks total)  
**Compute:** Kaggle CPU, Internet enabled  
**Status:** acquisition and inspection, not model training.

## What we are building
EDEN combines greenhouse climate forecasting, RGB-D crop estimation, physics-guided fusion and virtual control in a browser digital twin. This notebook establishes trustworthy source data. It does not train a model or claim predictive accuracy.

**Pipeline:** EDEN-00 → EDEN-01 climate ETL; EDEN-00 + EDEN-01 splits → EDEN-02 RGB-D pilot → later vision, physics, fusion and website modules.

## How to run
Import this `.ipynb` into Kaggle using the notebook import/upload option. Enable Internet, leave accelerator off, then execute cells in order. Read each short explanation, run its code, and inspect its output. Save a version with Run All after success. In the next notebook attach that saved output using Add Input. No laptop download of the dataset is needed.

## Deliverables
| File (under `/kaggle/working/eden`) | Purpose |
|---|---|
| `raw/README.md`, `raw/*timeseries.zip` | Verified official inputs for EDEN-01 |
| `source_manifest.json` | DOI, version, license, authors, URLs, checksums, acquisition status |
| `inventory.csv` | Archive member names and sizes |
| `schema_inventory.csv` | Observed CSV columns, sample types and missingness |
| `schema_map.json` | Candidate canonical mappings; unresolved entries explicitly marked |
| `reports/inventory_report.md` | Inspection summary and remaining blockers |
| `manifest.json` | Output hashes and handoff metadata |

**Completion gate:** numeric inputs verified and readable; schema candidates reviewed before ETL. RGB-D pair verification is deliberately deferred to EDEN-02 because the archive is 31.4 GB. A successful acquisition is not a full SRS inventory pass until unresolved metadata and mappings are reviewed.

## Cell 1 — Environment and directories
We use `pathlib` for paths, `requests` for HTTP, Pandas for tables, `zipfile` for archive inspection, and `hashlib` for integrity. Kaggle's working directory is writable; inputs attached later are read-only. The free disk check prevents avoidable failures. No GPU is needed for any cell here.

In [ ]:
from pathlib import Path
import csv, hashlib, io, json, platform, re, shutil, zipfile
from datetime import datetime, timezone
import pandas as pd
import requests
from IPython.display import display, Markdown

ROOT = Path('/kaggle/working/eden')
RAW = ROOT / 'raw'
REPORTS = ROOT / 'reports'
for p in (RAW, REPORTS):
    p.mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version(), '| Pandas:', pd.__version__)
print('Free disk GiB:', round(shutil.disk_usage(ROOT).free / 2**30, 2))
assert shutil.disk_usage(ROOT).free > 50 * 2**20, 'Insufficient disk for numeric pilot'
print('Working directory:', ROOT)

## Cell 2 — Pin the official source
A DOI identifies a research dataset; the versioned DOI makes our experiments reproducible. The supplied repository listing specifies CC BY 4.0: credit the authors, link the license and indicate modifications when sharing. MD5 checks against the publisher's fingerprints detect accidental corruption. SHA-256 will additionally identify our own handoff files. The image archive is recorded but not downloaded.

In [ ]:
BASE = 'https://data.4tu.nl/file/fa102772-32db-4b30-bace-12f2016722ce/'
SOURCES = {
    'README.md': dict(id='4cc2f511-9eb4-4cd3-99b9-97a6e106f3e1', size=3715, md5='f22125b584f3e25ae80ed87e2ba72de1', acquire=True),
    'autonomous_greenhouse_challenge4_timeseries.zip': dict(id='14a4a8cd-e081-4dbd-94e3-cd7b332ce120', size=6014133, md5='c4da1b2696c4a1e5ea5744f2482b0f93', acquire=True),
    'autonomous_greenhouse_challenge4_canopy_camera.zip': dict(id='818e2966-7015-432d-aa7c-1b9158c011cf', size=31416872168, md5='671a1c6e2b2e1627f821a36e431ceb28', acquire=False),
}
SOURCE = dict(title='4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images',
              doi='10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1', version=1,
              publisher='4TU.ResearchData', published='2025-06-23',
              authors=['Stef Maree','Pinglin Zhang','B. M. van Marrewijk','H.F. de Zwart','Monique Bijlaard','S. Hemming'],
              license='CC BY 4.0', license_url='https://creativecommons.org/licenses/by/4.0/',
              metadata_basis='Repository listing supplied by user; version 1', files=SOURCES)
for name, spec in SOURCES.items():
    spec['url'] = BASE + spec['id']
    spec['status'] = 'pending' if spec['acquire'] else 'deferred_to_EDEN-02'
    print(name, '|', spec['size'], '|', spec['status'])
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
save_json(ROOT / 'source_manifest.json', SOURCE)

## Cell 3 — Download directly into Kaggle and verify
HTTP requests retrieve the files into cloud storage. Streaming writes chunks rather than loading the full response into memory. A temporary `.part` file prevents an interrupted download from looking complete. We check both file length and MD5 before accepting it. Valid existing files are reused; if network access fails, enable Internet or attach an authorized copy via Add Input and rerun. The archive is never extracted wholesale.

In [ ]:
def digest(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def verified(path, spec):
    return path.is_file() and path.stat().st_size == spec['size'] and digest(path, 'md5') == spec['md5']

for name, spec in SOURCES.items():
    if not spec['acquire']:
        continue
    dest = RAW / name
    temp = dest.with_suffix(dest.suffix + '.part')
    try:
        if not verified(dest, spec):
            attached = next((p for p in Path('/kaggle/input').rglob(name) if verified(p, spec)), None)
            if attached:
                shutil.copyfile(attached, temp)
            else:
                with requests.get(spec['url'], stream=True, timeout=(20, 120)) as r:
                    r.raise_for_status()
                    total = 0
                    with temp.open('wb') as f:
                        for chunk in r.iter_content(1024*1024):
                            total += len(chunk)
                            if total > spec['size']:
                                raise ValueError('Response exceeds published size')
                            f.write(chunk)
            if not verified(temp, spec):
                raise ValueError('Size or MD5 mismatch; inspect source and connection')
            temp.replace(dest)
        spec.update(status='verified', sha256=digest(dest), local_path=str(dest.relative_to(ROOT)))
        print('VERIFIED:', name, dest.stat().st_size)
    except Exception as e:
        temp.unlink(missing_ok=True)
        spec.update(status='failed', error=str(e))
        raise
    finally:
        save_json(ROOT / 'source_manifest.json', SOURCE)
ARCHIVE = RAW / 'autonomous_greenhouse_challenge4_timeseries.zip'

## Cell 4 — Read the original documentation
The README is our authority for file meanings, units, timestamp conventions and experimental setup. A column called `temperature` alone does not tell us whether it is measured air temperature or a setpoint. Read this output before approving any mapping. We do not assume UTC, control semantics or depth calibration.

In [ ]:
readme = (RAW / 'README.md').read_text(encoding='utf-8-sig')
print(readme)

## Cell 5 — Inventory the numeric archive
ZIP central-directory metadata lists files without extracting them. We also check ZIP CRC integrity. The resulting table helps EDEN-01 locate climate, control, weather and harvest tables. Folder names may encode compartment identity; we preserve them exactly. The inventory below covers the numeric ZIP, not the deferred camera archive.

In [ ]:
with zipfile.ZipFile(ARCHIVE) as z:
    bad = z.testzip()
    assert bad is None, f'Corrupt ZIP member: {bad}'
    inventory = pd.DataFrame([dict(filename=m.filename, extension=Path(m.filename).suffix.lower(), size_bytes=m.file_size, compressed_bytes=m.compress_size, crc32=f'{m.CRC:08x}') for m in z.infolist() if not m.is_dir()])
assert not inventory.empty and inventory.filename.is_unique
inventory.to_csv(ROOT / 'inventory.csv', index=False)
display(inventory)
print('File types:')
print(inventory.extension.value_counts())

## Cell 6 — Inspect every CSV schema
A CSV is a text table. Delimiters can differ (comma, semicolon, tab); we infer them from a small sample and record what was used. We preview at most five rows per file and inspect up to 200 rows for sample types and missingness. These are preliminary observations, not full-table data quality measurements. Parsing errors are retained for review rather than silently dropped. No data are transformed here.

**Revision 1.1:** The ZIP contains AppleDouble metadata (`__MACOSX/`, `._*`) that can have misleading CSV extensions. We retain these in the raw archive inventory but exclude them before parsing, log the excluded paths, and check that all eight documented real CSV tables are present.

In [ ]:
schema_rows, inspections, errors = [], [], []
# Keep archive metadata in the inventory, but exclude it from data parsing.
all_csv_files = inventory.loc[inventory.extension.eq('.csv'), 'filename'].tolist()

def is_metadata(name):
    parts = Path(name).parts
    return '__MACOSX' in parts or any(part.startswith('.') for part in parts)

CSV_FILES = [name for name in all_csv_files if not is_metadata(name)]
SKIPPED_CSV_FILES = [name for name in all_csv_files if is_metadata(name)]
print('Genuine CSV files:', len(CSV_FILES))
print('Excluded metadata CSV files:', len(SKIPPED_CSV_FILES))
for name in SKIPPED_CSV_FILES:
    print('SKIP:', name)

# Version 1 contains six compartment tables and two weather tables.
expected_csv = {'agrifusion.csv', 'ideas.csv', 'mugrow.csv', 'reference.csv',
                'tomatonuts.csv', 'trigger.csv', 'weather.csv', 'weather_forecast.csv'}
assert len(CSV_FILES) == len(expected_csv) and {Path(name).name for name in CSV_FILES} == expected_csv, 'Unexpected genuine CSV inventory; review archive'
assert CSV_FILES, 'No CSV files found; inspect inventory manually'
with zipfile.ZipFile(ARCHIVE) as z:
    for name in CSV_FILES:
        try:
            with z.open(name) as f:
                raw = f.read(65536)
            encoding = 'utf-8-sig'
            try:
                text = raw.decode(encoding, errors='strict')
            except UnicodeDecodeError:
                encoding = 'cp1252'
                text = raw.decode(encoding)
            try:
                sep = csv.Sniffer().sniff(text, delimiters=',;\t|').delimiter
            except csv.Error:
                sep = None
            with z.open(name) as f:
                frame = pd.read_csv(f, sep=sep, engine='python', encoding=encoding, nrows=200)
            assert len(frame.columns) > 1, 'Single column: review delimiter/header'
            inspections.append(dict(filename=name, delimiter=sep, encoding=encoding, sample_rows=len(frame), columns=frame.columns.tolist()))
            for col in frame.columns:
                schema_rows.append(dict(filename=name, column=col, sample_dtype=str(frame[col].dtype), sample_missing=int(frame[col].isna().sum()), sample_rows=len(frame), example=str(frame[col].dropna().iloc[0]) if frame[col].notna().any() else ''))
            print('\nFILE:', name, '| delimiter:', repr(sep), '| sample shape:', frame.shape)
            display(frame.head(5))
        except Exception as e:
            errors.append(dict(filename=name, error=str(e)))
            print('REVIEW:', name, str(e))
schema = pd.DataFrame(schema_rows, columns=['filename','column','sample_dtype','sample_missing','sample_rows','example'])
schema.to_csv(ROOT / 'schema_inventory.csv', index=False)
save_json(REPORTS / 'csv_inspection.json', dict(files=inspections, errors=errors, excluded_metadata=SKIPPED_CSV_FILES))
print('Readable:', len(inspections), '/', len(CSV_FILES))

## Cell 7 — Propose canonical schema mappings (human review required)
EDEN uses stable names such as `air_temp_c` even if the original data uses another label. We search actual headers for possible matches; these are hints only, not accepted mappings. Multiple compartments or repeated headers are normal. A candidate must be checked against the README, units, measurement vs setpoint semantics and timestamp timezone. Missing fields remain unresolved. This cell deliberately does not create `climate.parquet` or assign splits; EDEN-01 owns those tasks.

In [ ]:
patterns = {
 'ts_utc': r'time|date|stamp',
 'compartment_id': r'compart|greenhouse|team|section',
 'air_temp_c': r'temp|tair|t_air',
 'rh_pct': r'humid|\brh\b|rh_',
 'co2_ppm': r'co.?2|carbon',
 'outside_temp_c': r'outside|outdoor|tout|t_out|external',
 'radiation_w_m2': r'radiat|solar|par|light',
 'ventilation': r'vent|window|roof|lee|windward',
 'heating': r'heat|pipe',
 'lighting': r'light|lamp|led',
 'co2_dosing': r'co.?2|dos',
 'irrigation': r'irrig|water|drip',
}
fields = {}
for target, pattern in patterns.items():
    hits = schema[schema.column.astype(str).str.contains(pattern, case=False, regex=True, na=False)][['filename','column']].to_dict('records')
    fields[target] = dict(status='candidate_review' if hits else 'unresolved', candidates=hits, approved_mapping=None, units=None)
    print(target, ':', len(hits), 'candidate(s)')
MAPPING = dict(schema_version='1.0', status='unreviewed', timezone=None, experiment_id='AGC4-2024', compartment_key_source=None, fields=fields, notes=['Candidate matches are lexical only; review README and actual samples.', 'Do not convert timestamps or treat setpoints as realized controls without confirmation.'])
save_json(ROOT / 'schema_map.json', MAPPING)
for key, value in fields.items():
    print('\n', key, ':', value['candidates'][:8])

## Cell 8 — Write inspection report and handoff manifest
A manifest connects each output to its source version and records SHA-256 hashes. The report distinguishes verified facts from unresolved questions. Future notebooks should attach this saved notebook output, verify hashes and read the schema map before converting data. The manifest cannot hash itself, so it hashes every other output file.

In [ ]:
report = f"""# EDEN-00 inventory report
- Source: {SOURCE['doi']} ({SOURCE['license']})
- Numeric archive members: {len(inventory)}
- Excluded metadata CSV files: {len(SKIPPED_CSV_FILES)}
- Genuine CSV files: {len(CSV_FILES)}; readable samples: {len(inspections)}; errors: {len(errors)}
- Source documentation and numeric ZIP: checksum verified
- Schema mapping: candidate only; human review required
- Timezone and compartment-key semantics: not yet approved
- RGB-D archive: deferred; no image pairs or calibration verified
- Next: review README, CSV samples and schema map, then implement EDEN-01
"""
(REPORTS / 'inventory_report.md').write_text(report, encoding='utf-8')
outputs = []
for p in sorted(ROOT.rglob('*')):
    if p.is_file() and p.name != 'manifest.json' and not p.name.endswith('.part'):
        outputs.append(dict(path=str(p.relative_to(ROOT)), size=p.stat().st_size, sha256=digest(p)))
manifest = dict(schema_version='1.0', producer='EDEN-00', code_version='1.1', source_doi=SOURCE['doi'], generated_utc=datetime.now(timezone.utc).isoformat(), schema_review='pending', image_inventory='deferred', csv_parse_errors=len(errors), files=outputs)
save_json(ROOT / 'manifest.json', manifest)
print(report)
print('Manifest entries:', len(outputs))

## Cell 9 — Acceptance gate
Assertions stop execution if required files are missing, checksums changed or CSV parsing failed. We intentionally report **numeric acquisition and inspection PASS** separately from the full handbook gate. A full gate also requires reviewed keys/units and sample RGB-D pairs; this notebook does not pretend those are complete. If a check fails, inspect the relevant earlier cell and rerun. A successful result is ready for schema review, not automatic model training.

In [ ]:
for name, spec in SOURCES.items():
    if spec['acquire']:
        assert verified(RAW / name, spec), name
for name in ['source_manifest.json','inventory.csv','schema_inventory.csv','schema_map.json','reports/inventory_report.md','manifest.json']:
    assert (ROOT / name).is_file(), name
assert not schema.empty, 'No schema observations'
assert not errors, f'CSV parsing needs review: {errors}'
assert {item['filename'] for item in inspections} == set(CSV_FILES), 'Missing genuine CSV inspections'
assert not schema.filename.map(is_metadata).any(), 'Metadata leaked into schema'
for entry in manifest['files']:
    assert digest(ROOT / entry['path']) == entry['sha256'], entry['path']
print('PASS: numeric acquisition and inspection')
print('PENDING: schema approval, timestamp/units review, RGB-D pair checks')
print('Next action: save notebook version and share README + CSV inspection output for EDEN-01.')

## What to do next
1. Review the README and CSV previews (cells 4 and 6). Identify actual climate, weather, control and harvest tables.
2. Share the cell 4, 6 and 9 output or the inspection JSON and schema inventory with your project guide. Approve mappings only after metadata review.
3. Save Version → Save & Run All; check the saved Output tab for `eden/`.
4. Create `EDEN-01-Climate-ETL` and attach this exact saved output version. That notebook will parse timestamps, map confirmed units, deduplicate, identify gaps and create leakage-safe compartment/time splits before model fitting.
5. Do not download the full canopy archive; EDEN-02 will investigate selective access and pairing after splits are fixed.

### Troubleshooting
- HTTP 403/timeout: confirm Kaggle Internet is enabled and the official URL remains accessible; use an authorized attached copy if necessary.
- MD5 mismatch: do not bypass checks; verify the version and publisher metadata.
- CSV parsing failure: inspect encoding, delimiter, header rows and README; update parser explicitly.
- No GPU: irrelevant for this CPU notebook.

### Attribution
Maree, S.; Zhang, P.; van Marrewijk, B. M.; de Zwart, H. F.; Bijlaard, M.; Hemming, S. (2025). *4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images*, v1. 4TU.ResearchData. https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1. CC BY 4.0. Numeric archive preserved unchanged; inventory and schema hints generated by EDEN.